In [ ]:
## LINK SHARED GOOGLE DRIVE
## Useful for the metadata nad .pth files, as well as saving downloads
from google.colab import drive
drive.mount('/content/drive')

import os

# Data Paths
SHARED_ROOT = "/content/drive/Shared drives/NPB 136 -- EYEBROW PIERCINGS"
STORAGE_FOLDER  = os.path.join(SHARED_ROOT, "Project File Storage")
MODEL_FOLDER = os.path.join(STORAGE_FOLDER, "Cell Dino pth Files")
os.makedirs(MODEL_FOLDER, exist_ok=True)
TENSOR_FOLDER = os.path.join(STORAGE_FOLDER, "Tensors")
os.makedirs(TENSOR_FOLDER, exist_ok=True)
# embeddings1: first run with no normalization
# embeddings2: 2nd run with naive normalization
EMBEDDINGS_FOLDER = os.path.join(STORAGE_FOLDER, "Embeddings4")
os.makedirs(EMBEDDINGS_FOLDER, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!du -sh "/content/drive/Shared drives/NPB 136 -- EYEBROW PIERCINGS/Project File Storage/Tensors"

4.0G	/content/drive/Shared drives/NPB 136 -- EYEBROW PIERCINGS/Project File Storage/Tensors


The model folder has the following files:
```
cell_dino_vits8_pretrain_cp-37d20e9c.pth
cell_dino_vitl16_pretrain_hpa_sc-d3ab8938.pth
cell_dino_vitl16_pretrain_hpa_fov-3505c6c0.pth
cell_dino_vitl14_pretrain_hpa_fov_highres-f57e7934.pth
channel_adaptive_dino_vitl16_pretrain_cells-ef7c17ff.pth
```

We want the channel adaptive one

In [ ]:
# install github repo if doesnt exist
![ -d "/content/dinov2" ] || git clone https://github.com/facebookresearch/dinov2

In [ ]:
import torch
import subprocess
import json
import torch.nn.functional as F

LOCAL_REPO = "/content/dinov2"

PATH_TO_DINO_PTH = os.path.join(MODEL_FOLDER, "channel_adaptive_dino_vitl16_pretrain_cells-ef7c17ff.pth")

model = torch.hub.load(
    LOCAL_REPO,
    'channel_adaptive_dino_vitl16',
    source='local',
    pretrained_path=PATH_TO_DINO_PTH
)

model.eval()
print("✓ Model loaded")
print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")

✓ Model loaded
  Params: 302,827,520


> MAKE SURE YOU HAVE GPU SELECTED

If it doesnt say `Using device: cuda` below, switch runtime:

On the ribbon above:
- Runtime -> Change Runtime Type -> (select GPU (A100 best, T4 if free account)) -> Save and restart session


In [ ]:
# Move model to GPU once
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = model.to(device)

def prepare_for_model(tensor, device):
    """
    tensor: (2, 600, 600)
    returns: (2, 1, 384, 384) — batch of single-channel inputs, on device
    """
    x = tensor.unsqueeze(1).float()  # (2, 1, 600, 600)
    x = F.interpolate(x, size=(384, 384), mode='bilinear', align_corners=False)
    return x.to(device)  # (2, 1, 384, 384)

# Test
meta = json.load(open(os.path.join(TENSOR_FOLDER, "metadata.json")))

test_tensor = torch.load(os.path.join(TENSOR_FOLDER, meta[0]["file"]))
channel_inputs = prepare_for_model(test_tensor, device)  # (2, 1, 384, 384)

with torch.no_grad():
    embeddings = model(channel_inputs)   # (2, 1024) — both channels in one forward pass
    features = embeddings.mean(dim=0, keepdim=True)  # (1, 1024)

print(f"✓ Forward pass OK")
print(f"  Input shape:         {channel_inputs.shape}")
print(f"  Embeddings shape:    {embeddings.shape}")
print(f"  Final feature shape: {features.shape}")

Using device: cuda
✓ Forward pass OK
  Input shape:         torch.Size([2, 1, 384, 384])
  Embeddings shape:    torch.Size([2, 1024])
  Final feature shape: torch.Size([1, 1024])


In [ ]:
from torch.utils.data import Dataset, DataLoader

class TensorDataset(Dataset):
    def __init__(self, meta, tensor_folder, device):
        self.meta = meta
        self.tensor_folder = tensor_folder
        self.device = device
        self.normalization = False

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        entry = self.meta[idx]
        tensor = torch.load(os.path.join(self.tensor_folder, entry["file"]))
        # Resize each channel: (2, 1, 384, 384)
        x = tensor.unsqueeze(1).float()
        if self.normalization:
          mean = x.mean(dim=[2, 3], keepdim=True)
          std  = x.std(dim=[2, 3],  keepdim=True).clamp(min=1e-8)# clamp guards agaisnt div by zero
          x    = (x - mean) / std
        x = F.interpolate(x, size=(384, 384), mode='bilinear', align_corners=False)
        label = entry["label"]
        return x, label, entry["patient_code"]

def collate_fn(batch):
    # x per item: (2, 1, 384, 384) → stack to (B*2, 1, 384, 384)
    xs      = torch.cat([item[0] for item in batch], dim=0)
    labels  = torch.tensor([item[1] for item in batch])
    codes   = [item[2] for item in batch]
    return xs, labels, codes

BATCH_SIZE = 8  # 8 files = 16 channel inputs per forward pass

dataset    = TensorDataset(meta, TENSOR_FOLDER, device)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_fn,
                        num_workers=2, pin_memory=True)

# Inference loop
all_features = []
all_labels   = []
all_codes    = []

model.eval()
with torch.no_grad():
    for batch_idx, (x, labels, codes) in enumerate(dataloader):
        x = x.to(device)                          # (B*2, 1, 384, 384)
        embeddings = model(x)                      # (B*2, 1024)

        # Reshape to (B, 2, 1024) then mean across channels → (B, 1024)
        B = labels.shape[0]
        features = embeddings.view(B, 2, -1).mean(dim=1)  # (B, 1024)

        all_features.append(features.cpu())
        all_labels.append(labels)
        all_codes.extend(codes)

        print(f"Batch {batch_idx+1}/{len(dataloader)} ✓  shape={features.shape}")

# Concatenate everything
all_features = torch.cat(all_features, dim=0)  # (N, 1024)
all_labels   = torch.cat(all_labels,   dim=0)  # (N,)

print(f"\n✓ Done")
print(f"  Features: {all_features.shape}")
print(f"  Labels:   {all_labels.shape}")

Batch 1/91 ✓  shape=torch.Size([8, 1024])
Batch 2/91 ✓  shape=torch.Size([8, 1024])
Batch 3/91 ✓  shape=torch.Size([8, 1024])
Batch 4/91 ✓  shape=torch.Size([8, 1024])
Batch 5/91 ✓  shape=torch.Size([8, 1024])
Batch 6/91 ✓  shape=torch.Size([8, 1024])
Batch 7/91 ✓  shape=torch.Size([8, 1024])
Batch 8/91 ✓  shape=torch.Size([8, 1024])
Batch 9/91 ✓  shape=torch.Size([8, 1024])
Batch 10/91 ✓  shape=torch.Size([8, 1024])
Batch 11/91 ✓  shape=torch.Size([8, 1024])
Batch 12/91 ✓  shape=torch.Size([8, 1024])
Batch 13/91 ✓  shape=torch.Size([8, 1024])
Batch 14/91 ✓  shape=torch.Size([8, 1024])
Batch 15/91 ✓  shape=torch.Size([8, 1024])
Batch 16/91 ✓  shape=torch.Size([8, 1024])
Batch 17/91 ✓  shape=torch.Size([8, 1024])
Batch 18/91 ✓  shape=torch.Size([8, 1024])
Batch 19/91 ✓  shape=torch.Size([8, 1024])
Batch 20/91 ✓  shape=torch.Size([8, 1024])
Batch 21/91 ✓  shape=torch.Size([8, 1024])
Batch 22/91 ✓  shape=torch.Size([8, 1024])
Batch 23/91 ✓  shape=torch.Size([8, 1024])
Batch 24/91 ✓  shape

In [ ]:
print(all_features)
print(f"  Features: {all_features.shape}")
print(all_labels)
print(f"  Labels:   {all_labels.shape}")
torch.save({
    "features":      all_features,       # (300, 1024)
    "labels":        all_labels,          # (300,)
    "patient_codes": all_codes,           # list of 300 strings
    "label_map":     {"PD": 1, "HC": 0},
    "feature_dim":   all_features.shape[1],
    "n_samples":     all_features.shape[0],
    "model":         "channel_adaptive_dino_vitl16",
}, os.path.join(EMBEDDINGS_FOLDER, "embeddings.pt"))

print(f"✓ Saved to {EMBEDDINGS_FOLDER}/embeddings.pt")

tensor([[-0.6020, -0.2991,  2.4978,  ...,  0.6175, -1.9729, -0.1314],
        [-0.0865, -0.0730,  2.3761,  ...,  0.9214, -2.1964, -0.0563],
        [-0.2351, -0.2607,  3.0174,  ...,  0.4663, -1.6323, -0.3074],
        ...,
        [-0.4279, -0.4638,  2.8715,  ...,  0.5610, -1.9113, -0.3152],
        [ 0.3534, -0.4153,  2.2107,  ...,  1.0438, -2.4258, -0.2623],
        [ 0.0569, -0.1568,  1.9244,  ...,  1.1388, -2.4711, -0.2987]])
  Features: torch.Size([727, 1024])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [ ]:
!du -sh "/content/drive/Shared drives/NPB 136 -- EYEBROW PIERCINGS/Project File Storage/Embeddings3"

3.1M	/content/drive/Shared drives/NPB 136 -- EYEBROW PIERCINGS/Project File Storage/Embeddings3
